### Gold: `03_gold.zsi_windows` — P, D and ZSI for every physical zone × morning window
- P = pickups in zone z whose pickup_ts falls in window w; D = dropoffs in z whose dropoff_ts falls in w
- ZSI = √P + √(P+1) − √(4D+3)
- Every physical zone gets all 210 morning windows (P = D = 0 when there were no trips)

In [0]:
%run ../../config/config

In [0]:
%run ../../utils/zsi_functions

In [0]:
dbutils.widgets.text("settings", "spec")
s = ZSI_SETTINGS[dbutils.widgets.get("settings")]
suffix = s["table_suffix"]
op_day_start, window_min = hhmm_to_min(s["op_day_start"]), int(s["window_minutes"])

In [0]:
from pyspark.sql.functions import broadcast, col, count, expr, floor, hour, lit, minute, sqrt, to_date


def window_key(ts_col):
    """(operational_date, window_idx) of a wall-clock timestamp."""
    shifted = col(ts_col) - expr(f"INTERVAL {op_day_start} MINUTES")
    return (to_date(shifted).alias("operational_date"),
            floor((hour(shifted) * 60 + minute(shifted)) / window_min).cast("int").alias("window_idx"))


fact = spark.table(tbl("02_silver", "fact_trips"))
zones = spark.table(tbl("02_silver", "taxi_zone_lookup")).filter("is_physical").select(col("location_id").alias("LocationID"))
morning = spark.table(tbl("03_gold", f"window_grid{suffix}")).filter("is_am") \
               .select("operational_date", "window_idx", "window_start", "window_start_tod")

P = fact.select(col("pu_location_id").alias("LocationID"), *window_key("pickup_ts")) \
        .groupBy("LocationID", "operational_date", "window_idx").agg(count("*").alias("P"))
D = fact.select(col("do_location_id").alias("LocationID"), *window_key("dropoff_ts")) \
        .groupBy("LocationID", "operational_date", "window_idx").agg(count("*").alias("D"))

keys = ["LocationID", "operational_date", "window_idx"]
zsi_windows = (
    zones.crossJoin(broadcast(morning))
    .join(P, keys, "left").join(D, keys, "left")
    .fillna(0, subset=["P", "D"])
    .withColumn("ZSI", sqrt(col("P").cast("double")) + sqrt(col("P") + lit(s["p_offset"]))
                - sqrt(lit(4.0) * col("D") + lit(s["smoothing_constant"])))
    .select(*keys, "window_start", "window_start_tod", col("P").cast("long"), col("D").cast("long"), "ZSI")
)
zsi_windows.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("03_gold", f"zsi_windows{suffix}"))

In [0]:
check = spark.sql(f"""
SELECT COUNT(DISTINCT LocationID) AS zones, MIN(n) AS min_windows, MAX(n) AS max_windows
FROM (SELECT LocationID, COUNT(*) AS n FROM {tbl('03_gold', f'zsi_windows{suffix}')} GROUP BY LocationID)""").first()
print(check)
if s["expected_am_windows"] is not None:
    assert check.min_windows == check.max_windows == s["expected_am_windows"], "every zone must have exactly 210 windows"